In [19]:
!mkdir yolov3

In [21]:
%%writefile yolov3/dataset.py
import pandas as pd
import numpy as np
import torch
import os
from utils import intersection_over_union
class YOLODataset(torch.utils.data.Dataset):
    def __init__(self, csv_file, img_dir, label_dir, anchors,img_size = 416, transform = None, S = [13,26,52], C = 20):
        self.annotations = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.anchors = (anchors[0] + anchors[1] + anchors[2])
        self.img_size = img_size
        self.S = S
        self.transform = transform
        self.num_anchors = self.anchors.shape[0]
        self.num_anchors_per_scale = self.num_anchors // 3
        self.C = C
        self.iou_threshold = 0.5
        
    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        label_path = os.path.join([self.label_dir, self.annotations[index, 0]])
        boxes = np.roll(np.loadtxt(f = label_path, delimiter = '', ndmin = 2), 4, axis = 1) # to be [conf_score,x,y,w,h,class_label]
        img_path = os.path.join([self.img_dir, self.annotations[index, 1]])
        img = np.array(Image.open(img_path).convert('RGB'))

        if self.transform:
            augmentations = self.transform(img, boxes)
            img = augmentations['img']
            boxes = augmentations['boxes']

        target = [torch.zeros((self.num_anchors // 3, S, S, 6)) for S in self.S]
        """
        targets = [
        [3,13,13,6]
        [3,26,26,6]
        [3,52,52,6]
        ]
        represents each scale output as required by the paper 
        now we have to map boxes and img to correspond to this mapping
        """
        for box in boxes:
            iou = intersection_over_union(box[2:4], self.anchors)
            anchor_indices = iou.argsort(descending = True, dim = 0)
            x, y, w, h, class_label = box
            has_anchor = [False]*3
            """
            anchor0 = for large scale (13x13) == False
            anchor1 = for medium scale (26x26) == False
            anchor2 = for small scale (52x52) == False
            """

            for anchor_idx in anchor_indices:
                scale_idx = anchor_idx // self.num_anchors_per_scale 
                """
                targets = [
                [3,13,13,6] == scale_idx = 0
                [3,26,26,6] == scale_idx = 1
                [3,52,52,6] == scale_idx = 2
                ]
                """
                anchor_scale = anchor_idx % self.num_anchors_per_scale # this returns remainder which indicate on that specific scale which anchor to use
                #now to convert according to rows and columns for the scale to which we have got
                S = self.anchors[scale_idx]
                i, j = int(S*y), int(S*x) # find where the object center is by calculating the row and column of it in the scale e.g object center is found on 3 row 2 column
                anchor_taken = target[scale_idx][anchor_scale, i, j, 0]
                if not anchor_taken and has_anchor[scale_idx]:
                    target[scale_idx][anchor_scale, i, j, 0] = 1
                    x_cell , y_cell = S*x -j, S*y - i
                    w_cell, h_cell = S*w, S*h
                    box_cordinates = torch.tensor(x_cell, y_cell, w_cell, h_cell)
                    target[scale_idx][anchor_scale, i, j, 1:5] = box_cordinates
                    
                    target[scale_idx][anchor_scale, i, j, 5] = int(class_label) 

                    has_anchor[scale_idx] = True

                elif not anchor_taken and iou[anchor_idx] > self.iou_threshold:
                    target[scale_idx][anchor_scale, i, j, 0] = -1

        return img, tuple(target)
        
"""
so the idea is that we have our dataset already arranged in order of conf_score,x,y,w,h, class_label but the model expects it's form inorder to train with this
so, we have to convert these true labels to correspond to the model output that is
1.firstly, we initiate the target to be as,
 targets = [
        [3,13,13,6]
        [3,26,26,6]
        [3,52,52,6]
        ]
 =>  3 - represents three anchor boxes per every scale, 13 - the scale itself, 6 - contains the dataset 
 
 2. we loop through all the boxes in the dataset, then we find intersection between every box and anchors so as to get anchors that could be used for bounding boxes
 we arrange from the largest to the smallest ===== everything is explained in the code
 3. 
 
"""
        

Writing yolov3/dataset.py


In [23]:
%%writefile yolov3/loss.py
from torch import nn
class YoloLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.mse = nn.MSELoss()
        self.entropy = nn.CrossEntropyLoss()
        self.sigmoid = nn.Sigmoid()

        self.lambda_noobj = 10
        self.lambda_obj = 1
        self.lambda_class = 1
        self.lambda_box = 10

    def forward(self, predictions, target, anchors):

        obj = target[..., 0] == 1
        noobj = target[..., 0] == 0

        no_object_loss = self.bce(
            predictions[..., 0:1][noobj], target[..., 0:1][noobj]
        )
        
        anchors = anchors.reshape(1,3,1,1,2)
        box_pred = torch.cat(self.sigmoid(predictions[...,1:3]), torch.exp(predictions[3:5] * anchors), dim = -1)
        iou = intersection_over_union(box_pred[obj], target[..., 1:5][obj]).detach()
        object_loss = self.mse(self.sigmoid(predictions[..., 0:1][obj]), iou * target[..., 0:1][obj])    

        predictions[..., 1:3] = self.sigmoid(predictions[...,1:3])
        target[..., 3:5] = torch.log(1e-16 + target[...,3:5]/anchors)
        box_loss = self.mse(
            predictions[..., 1:5][obj], target[..., 1:5][obj]
        )

        class_loss = self.entropy(
            predictions[..., 5][obj], target[..., 5][obj].long()
        )

        loss = (
            self.lambda_obj * obj_loss +
            self.lamda_noobj * no_object_loss +
            self.lambda_class * class_loss +
            self.lambda_box * box_loss
        )
        return loss
        

Writing yolov3/loss.py


In [1]:
%%writefile yolov3/main.py
from loss import YoloLoss
from utils import (
    mean_average_precision,
    cells_to_bboxes,
    get_evaluation_bboxes,
    save_checkpoint,
    load_checkpoint,
    check_class_accuracy,
    get_loaders,
    plot_couple_examples
)
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
torch.backends.cudnn.benchmark = True

def train_fn(dataloader, model, loss_fn, optimizer, scaler, scaler_anchors, device):
    losses = []
    loop = tqdm(dataloader, leave = True)
    for batch, (x,y) in enumerate(loop):
        
        x = x.to(device)
        y0, y1, y2 = (
            y[0].to(device),
            y[1].to(device),
            y[2].to(device)
        )
        with.torch.cuda.amp.autocast()
        out = model(x)
        for idx in range(len(out)):
            loss += loss_fn(out[idx], y[idx], scaler_anchors[idx])
        losses.append(loss)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        mean_loss = sum(losses) / len(losses)
        loop.set_postfix(loss = mean_loss)

def main():
    model = YOLOv3(num_classes=config.NUM_CLASSES).to(config.DEVICE)
    optimizer = optim.Adam(
        model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY
    )
    loss_fn = YoloLoss()
    scaler = torch.cuda.amp.GradScaler()

    train_loader, test_loader, train_eval_loader = get_loaders(
        train_csv_path=config.DATASET + "/train.csv", test_csv_path=config.DATASET + "/test.csv"
    )

    if config.LOAD_MODEL:
        load_checkpoint(
            config.CHECKPOINT_FILE, model, optimizer, config.LEARNING_RATE
        )

    scaled_anchors = (
        torch.tensor(config.ANCHORS)
        * torch.tensor(config.S).unsqueeze(1).unsqueeze(1).repeat(1, 3, 2)
    ).to(config.DEVICE)

    for epoch in range(config.NUM_EPOCHS): 
        train_fn(train_loader, model, optimizer, loss_fn, scaler, scaled_anchors)
        
        if epoch > 0 and epoch % 3 == 0:
            check_class_accuracy(model, test_loader, threshold=config.CONF_THRESHOLD)
            pred_boxes, true_boxes = get_evaluation_bboxes(
                test_loader,
                model,
                iou_threshold=config.NMS_IOU_THRESH,
                anchors=config.ANCHORS,
                threshold=config.CONF_THRESHOLD,
            )
            mapval = mean_average_precision(
                pred_boxes,
                true_boxes,
                iou_threshold=config.MAP_IOU_THRESH,
                box_format="midpoint",
                num_classes=config.NUM_CLASSES,
            )
            print(f"MAP: {mapval.item()}")
            model.train()


if __name__ == "__main__":
    main()

Writing yolov3/main.py


In [13]:
%%writefile yolov3/utils.py
def intersection_over_union(target, predictions, box_format = 'mid-point'):
    """
    given that midpoint represent [class_label, conf_score, x, y, w, h]
    given that corners represent [...x1, y1, x2, y2]
    """
    if box_format == 'mid-point':
        box1_x1 = predictions[..., 21:22] - predictions[..., 23:24]/2
        box1_y1 = predictions[..., 22:23] - predictions[..., 24:25]/2
        box1_x2 = predictions[..., 21:22] + predictions[..., 23:24]/2
        box1_y2 = predictions[..., 22:23] + predictions[..., 23:24]/2
        
        box2_x1 = target[..., 26:27] - target[..., 28:29]/2
        box2_y1 = target[..., 27:28] - target[..., 29:30]/2
        box2_x2 = target[..., 26:27] + target[..., 28:29]/2
        box2_y2 = target[..., 27:28] + target[..., 29:30]/2
    if box_format == 'corners':
        box1_x1 = predictions[..., 21:22]
        box1_y1 = predictions[..., 22:23]
        box1_x2 = predictions[..., 23:24]
        box1_y2 = predictions[..., 24:25]
        
        box1_x1 = target[..., 21:22]
        box1_y1 = target[..., 22:23]
        box1_x2 = target[..., 23:24]
        box1_y2 = target[..., 24:25]

    x1 = torch.max(box1_x1, box2_x1)
    y1 = torch.max(box1_y1, box2_y1)
    x2 = torch.min(box1_x2, box2_x2)
    y2 = torch.min(box1_y2, box2_y2)

    intersection = ((x2-x1).clamp(0)) * ((y2-y1).clamp(0))
    area_b1 = abs((box1_x1 - box1_x2) * (box1_y1 - box1_y2))
    area_b2 = abs((box2_x1 - box2_x2) * (box2_y1 - box2_y2))

    iou = intersection / (area_b1 + area_b2 - intersection + 1e-6)
    
    return iou
def iou_width_height(boxes1, boxes2):
    """
    Parameters:
        boxes1 (tensor): width and height of the first bounding boxes
        boxes2 (tensor): width and height of the second bounding boxes
    Returns:
        tensor: Intersection over union of the corresponding boxes
    """
    intersection = torch.min(boxes1[..., 0], boxes2[..., 0]) * torch.min(
        boxes1[..., 1], boxes2[..., 1]
    )
    union = (
        boxes1[..., 0] * boxes1[..., 1] + boxes2[..., 0] * boxes2[..., 1] - intersection
    )
    return intersection / union

def non_max_suppresion(bboxes, threshold, iou_threshold, box_format = 'corners'):
    assert type(bboxes) == list
    bboxes = [box for box in bboxes if box[1]>threshold ]

    bboxes = sorted(bboxes, key = lambda x: x[1], reverse = True)
    bboxes_after_nms = []
    while bboxes:
        chosen_box = bboxes.pop(0)

        bboxes = [
            box
            for box in bboxes
            if chosen_box[0] != box[0] or
            intersection_over_union(
                torch.tensor(box[2:]),
                torch.tensor(chosen_box[2:])
                box_format = box_format
            )
        ]
        bboxes_after_nms.append(chosen_box)
    return bboxes_after_nms


def plot_image(image, boxes):
    """Plots predicted bounding boxes on the image"""
    cmap = plt.get_cmap("tab20b")
    class_labels = config.COCO_LABELS if config.DATASET=='COCO' else config.PASCAL_CLASSES
    colors = [cmap(i) for i in np.linspace(0, 1, len(class_labels))]
    im = np.array(image)
    height, width, _ = im.shape

    # Create figure and axes
    fig, ax = plt.subplots(1)
    # Display the image
    ax.imshow(im)

    # box[0] is x midpoint, box[2] is width
    # box[1] is y midpoint, box[3] is height

    # Create a Rectangle patch
    for box in boxes:
        assert len(box) == 6, "box should contain class pred, confidence, x, y, width, height"
        class_pred = box[0]
        box = box[2:]
        upper_left_x = box[0] - box[2] / 2
        upper_left_y = box[1] - box[3] / 2
        rect = patches.Rectangle(
            (upper_left_x * width, upper_left_y * height),
            box[2] * width,
            box[3] * height,
            linewidth=2,
            edgecolor=colors[int(class_pred)],
            facecolor="none",
        )
        # Add the patch to the Axes
        ax.add_patch(rect)
        plt.text(
            upper_left_x * width,
            upper_left_y * height,
            s=class_labels[int(class_pred)],
            color="white",
            verticalalignment="top",
            bbox={"color": colors[int(class_pred)], "pad": 0},
        )

    plt.show()


def get_evaluation_bboxes(
    loader,
    model,
    iou_threshold,
    anchors,
    threshold,
    box_format="midpoint",
    device="cuda",
):
    # make sure model is in eval before get bboxes
    model.eval()
    train_idx = 0
    all_pred_boxes = []
    all_true_boxes = []
    for batch_idx, (x, labels) in enumerate(tqdm(loader)):
        x = x.to(device)

        with torch.no_grad():
            predictions = model(x)

        batch_size = x.shape[0]
        bboxes = [[] for _ in range(batch_size)]
        for i in range(3):
            S = predictions[i].shape[2]
            anchor = torch.tensor([*anchors[i]]).to(device) * S
            boxes_scale_i = cells_to_bboxes(
                predictions[i], anchor, S=S, is_preds=True
            )
            for idx, (box) in enumerate(boxes_scale_i):
                bboxes[idx] += box

        # we just want one bbox for each label, not one for each scale
        true_bboxes = cells_to_bboxes(
            labels[2], anchor, S=S, is_preds=False #ground truth is easiest to extract from highest resolution scale
        )

        for idx in range(batch_size):
            nms_boxes = non_max_suppression(
                bboxes[idx],
                iou_threshold=iou_threshold,
                threshold=threshold,
                box_format=box_format,
            )

            for nms_box in nms_boxes:
                all_pred_boxes.append([train_idx] + nms_box)

            for box in true_bboxes[idx]:
                if box[1] > threshold:
                    all_true_boxes.append([train_idx] + box)

            train_idx += 1

    model.train()
    return all_pred_boxes, all_true_boxes


def cells_to_bboxes(predictions, anchors, S, is_preds=True):
    """
    Scales the predictions coming from the model to
    be relative to the entire image such that they for example later
    can be plotted or.
    INPUT:
    predictions: tensor of size (N, 3, S, S, num_classes+5)
    anchors: the anchors used for the predictions
    S: the number of cells the image is divided in on the width (and height)
    is_preds: whether the input is predictions or the true bounding boxes
    OUTPUT:
    converted_bboxes: the converted boxes of sizes (N, num_anchors, S, S, 1+5) with class index,
                      object score, bounding box coordinates
    """
    BATCH_SIZE = predictions.shape[0]
    num_anchors = len(anchors)
    box_predictions = predictions[..., 1:5]
    if is_preds:
        anchors = anchors.reshape(1, len(anchors), 1, 1, 2)
        box_predictions[..., 0:2] = torch.sigmoid(box_predictions[..., 0:2])
        box_predictions[..., 2:] = torch.exp(box_predictions[..., 2:]) * anchors
        scores = torch.sigmoid(predictions[..., 0:1])
        best_class = torch.argmax(predictions[..., 5:], dim=-1).unsqueeze(-1)
    else:
        scores = predictions[..., 0:1]
        best_class = predictions[..., 5:6]

    cell_indices = (
        torch.arange(S)
        .repeat(predictions.shape[0], 3, S, 1)
        .unsqueeze(-1)
        .to(predictions.device)
    )
    x = 1 / S * (box_predictions[..., 0:1] + cell_indices)
    y = 1 / S * (box_predictions[..., 1:2] + cell_indices.permute(0, 1, 3, 2, 4))
    w_h = 1 / S * box_predictions[..., 2:4]
    converted_bboxes = torch.cat((best_class, scores, x, y, w_h), dim=-1).reshape(BATCH_SIZE, num_anchors * S * S, 6)
    return converted_bboxes.tolist()

def check_class_accuracy(model, loader, threshold):
    model.eval()
    tot_class_preds, correct_class = 0, 0
    tot_noobj, correct_noobj = 0, 0
    tot_obj, correct_obj = 0, 0

    for idx, (x, y) in enumerate(tqdm(loader)):
        x = x.to(config.DEVICE)
        with torch.no_grad():
            out = model(x)

        for i in range(3):
            y[i] = y[i].to(config.DEVICE)
            obj = y[i][..., 0] == 1 # in paper this is Iobj_i
            noobj = y[i][..., 0] == 0  # in paper this is Iobj_i

            correct_class += torch.sum(
                torch.argmax(out[i][..., 5:][obj], dim=-1) == y[i][..., 5][obj]
            )
            tot_class_preds += torch.sum(obj)

            obj_preds = torch.sigmoid(out[i][..., 0]) > threshold
            correct_obj += torch.sum(obj_preds[obj] == y[i][..., 0][obj])
            tot_obj += torch.sum(obj)
            correct_noobj += torch.sum(obj_preds[noobj] == y[i][..., 0][noobj])
            tot_noobj += torch.sum(noobj)

    print(f"Class accuracy is: {(correct_class/(tot_class_preds+1e-16))*100:2f}%")
    print(f"No obj accuracy is: {(correct_noobj/(tot_noobj+1e-16))*100:2f}%")
    print(f"Obj accuracy is: {(correct_obj/(tot_obj+1e-16))*100:2f}%")
    model.train()


def get_mean_std(loader):
    # var[X] = E[X**2] - E[X]**2
    channels_sum, channels_sqrd_sum, num_batches = 0, 0, 0

    for data, _ in tqdm(loader):
        channels_sum += torch.mean(data, dim=[0, 2, 3])
        channels_sqrd_sum += torch.mean(data ** 2, dim=[0, 2, 3])
        num_batches += 1

    mean = channels_sum / num_batches
    std = (channels_sqrd_sum / num_batches - mean ** 2) ** 0.5

    return mean, std

Writing yolov3/utils.py


In [31]:
import torch
import torch.nn as nn

""" 
Information about architecture config:
Tuple is structured by (filters, kernel_size, stride) 
Every conv is a same convolution. 
List is structured by "B" indicating a residual block followed by the number of repeats
"S" is for scale prediction block and computing the yolo loss
"U" is for upsampling the feature map and concatenating with a previous layer
"""
config = [
    (32, 3, 1),
    (64, 3, 2),
    ["B", 1],
    (128, 3, 2),
    ["B", 2],
    (256, 3, 2),
    ["B", 8],
    (512, 3, 2),
    ["B", 8],
    (1024, 3, 2),
    ["B", 4],  # To this point is Darknet-53
    (512, 1, 1),
    (1024, 3, 1),
    "S",
    (256, 1, 1),
    "U",
    (256, 1, 1),
    (512, 3, 1),
    "S",
    (128, 1, 1),
    "U",
    (128, 1, 1),
    (256, 3, 1),
    "S",
]


class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, bn_act=True, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, bias=not bn_act, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)
        self.leaky = nn.LeakyReLU(0.1)
        self.use_bn_act = bn_act

    def forward(self, x):
        if self.use_bn_act:
            return self.leaky(self.bn(self.conv(x)))
        else:
            return self.conv(x)


class ResidualBlock(nn.Module):
    def __init__(self, channels, use_residual=True, num_repeats=1):
        super().__init__()
        self.layers = nn.ModuleList()
        for repeat in range(num_repeats):
            self.layers += [
                nn.Sequential(
                    CNNBlock(channels, channels // 2, kernel_size=1),
                    CNNBlock(channels // 2, channels, kernel_size=3, padding=1),
                )
            ]

        self.use_residual = use_residual
        self.num_repeats = num_repeats

    def forward(self, x):
        for layer in self.layers:
            if self.use_residual:
                x = x + layer(x)
            else:
                x = layer(x)

        return x


class ScalePrediction(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.pred = nn.Sequential(
            CNNBlock(in_channels, 2 * in_channels, kernel_size=3, padding=1),
            CNNBlock(
                2 * in_channels, (num_classes + 5) * 3, bn_act=False, kernel_size=1
            ),
        )
        self.num_classes = num_classes

    def forward(self, x):
        return (
            self.pred(x)
            .reshape(x.shape[0], 3, self.num_classes + 5, x.shape[2], x.shape[3])
            .permute(0, 1, 3, 4, 2)
        )


class YOLOv3(nn.Module):
    def __init__(self, in_channels=3, num_classes=80):
        super().__init__()
        self.num_classes = num_classes
        self.in_channels = in_channels
        self.layers = self._create_conv_layers()

    def forward(self, x):
        outputs = []  # for each scale
        route_connections = []
        for layer in self.layers:
            if isinstance(layer, ScalePrediction):
                outputs.append(layer(x))
                continue

            x = layer(x)

            if isinstance(layer, ResidualBlock) and layer.num_repeats == 8:
                route_connections.append(x)

            elif isinstance(layer, nn.Upsample):
                x = torch.cat([x, route_connections[-1]], dim=1)
                route_connections.pop()

        return outputs

    def _create_conv_layers(self):
        layers = nn.ModuleList()
        in_channels = self.in_channels

        for module in config:
            if isinstance(module, tuple):
                out_channels, kernel_size, stride = module
                layers.append(
                    CNNBlock(
                        in_channels,
                        out_channels,
                        kernel_size=kernel_size,
                        stride=stride,
                        padding=1 if kernel_size == 3 else 0,
                    )
                )
                in_channels = out_channels

            elif isinstance(module, list):
                num_repeats = module[1]
                layers.append(ResidualBlock(in_channels, num_repeats=num_repeats,))

            elif isinstance(module, str):
                if module == "S":
                    layers += [
                        ResidualBlock(in_channels, use_residual=False, num_repeats=1),
                        CNNBlock(in_channels, in_channels // 2, kernel_size=1),
                        ScalePrediction(in_channels // 2, num_classes=self.num_classes),
                    ]
                    in_channels = in_channels // 2

                elif module == "U":
                    layers.append(nn.Upsample(scale_factor=2),)
                    in_channels = in_channels * 3

        return layers


if __name__ == "__main__":
    num_classes = 20
    IMAGE_SIZE = 416
    model = YOLOv3(num_classes=num_classes)
    x = torch.randn((2, 3, IMAGE_SIZE, IMAGE_SIZE))
    out = model(x)
    assert model(x)[0].shape == (2, 3, IMAGE_SIZE//32, IMAGE_SIZE//32, num_classes + 5)
    assert model(x)[1].shape == (2, 3, IMAGE_SIZE//16, IMAGE_SIZE//16, num_classes + 5)
    assert model(x)[2].shape == (2, 3, IMAGE_SIZE//8, IMAGE_SIZE//8, num_classes + 5)
    print("Success!")

Success!


In [33]:
sample = YOLOv3()
sample

YOLOv3(
  (layers): ModuleList(
    (0): CNNBlock(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (leaky): LeakyReLU(negative_slope=0.1)
    )
    (1): CNNBlock(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (leaky): LeakyReLU(negative_slope=0.1)
    )
    (2): ResidualBlock(
      (layers): ModuleList(
        (0): Sequential(
          (0): CNNBlock(
            (conv): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (leaky): LeakyReLU(negative_slope=0.1)
          )
          (1): CNNBlock(
            (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=

In [11]:
%%writefile yolov3/model.py
import torch
from torch import nn
config = [
    (32, 3, 1),
    (64, 3, 2),
    ["B", 1],
    (128, 3, 2),
    ["B", 2],
    (256, 3, 2),
    ["B", 8],
    (512, 3, 2),
    ["B", 8],
    (1024, 3, 2),
    ["B", 4],  # To this point is Darknet-53
    (512, 1, 1),
    (1024, 3, 1),
    "S",
    (256, 1, 1),
    "U",
    (256, 1, 1),
    (512, 3, 1),
    "S",
    (128, 1, 1),
    "U",
    (128, 1, 1),
    (256, 3, 1),
    "S",
]

class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels,bn_act = True, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, bias = not bn_act, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)
        self.leaky = nn.LeakyReLU(0.1)
        self.use_bn = bn_act
    def forward(self, x):
        if self.use_bn:
            x = self.leaky(self.bn(self.conv(x)))
        else:
            x = self.conv(x)
        return x

class ResidualBlock(nn.Module):
    def __init__(self, channels, use_residual = True, num_repeats = 1):
        super().__init__()
        self.layers = nn.ModuleList()
        self.num_repeats = num_repeats
        for _ in range(self.num_repeats):
            self.layers += [
                nn.Sequential(
                    CNNBlock(channels, channels//2, kernel_size = 1),
                    CNNBlock(channels //2 , channels, kernel_size = 3, padding = 1)
                )
            ]
        self.use_residual = use_residual
    def forward(self, x):
        for layers in self.layers:
            if self.use_residual:
                x = x + layers(x)
            else:
                x = layers(x)
        return x

class ScalePredictions(nn.Module):
    def __init__(self,in_channels, num_classes = 80):
        super().__init__()
        self.pred = nn.Sequential(
            CNNBlock(in_channels, in_channels * 2, kernel_size = 3, padding = 1),
            CNNBlock(in_channels * 2, (num_classes + 5) * 3, bn_act = False, kernel_size = 1)
        )
        self.num_classes = num_classes
    def forward(self, x):
        x = self.pred(x).reshape(x.shape[0], 3, (self.num_classes+5), x.shape[2], x.shape[3]).permute(0, 1, 3, 4, 2)
        return x
#(filters, kernel_size, stride) 
class YOLOv3(nn.Module):
    def __init__(self, in_channels = 3, num_classes = 80):
        super().__init__()
        self.config = config
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.conv_layers = self._conv_layers()

    def forward(self,x):
        output = []
        root_connections = []
        for layer in self.conv_layers:
            if isinstance(layer, ScalePredictions):
                output.append(layer(x))
                continue
                
            x = layer(x)

            if isinstance(layer, ResidualBlock) and layer.num_repeats == 8:
                root_connections.append(x)

            elif isinstance(layer, nn.Upsample):
                x = torch.cat([x, root_connections[-1]], dim=1)
                root_connections.pop()
        return output
            
    def _conv_layers(self):
        layers = nn.ModuleList()
        in_channels = self.in_channels
        for module in self.config:
            if isinstance(module, tuple):
                out_channels, kernel_size, stride = module
                layers.append(
                    CNNBlock(in_channels, out_channels, kernel_size = kernel_size, stride = stride, padding = 1 if kernel_size == 3 else 0)
                )
                in_channels = out_channels
            elif isinstance(module, list):
                num_repeat = module[1]
                layers.append(
                    ResidualBlock(in_channels, num_repeats = num_repeat)
                )
            elif isinstance(module, str):
                if module == 'S':
                    layers +=[
                        ResidualBlock(in_channels, use_residual = False, num_repeats = 1),
                        CNNBlock(in_channels, in_channels // 2, bn_act = False, kernel_size = 1),
                        ScalePredictions(in_channels//2, num_classes = self.num_classes)
                    ]
                    in_channels = in_channels // 2
                elif module == 'U':
                    layers.append(
                        nn.Upsample(scale_factor = 2)
                    )
                    in_channels = in_channels * 3 # in_channels = channels_from_upsample + channels_from_route
        return layers

if __name__ == "__main__":
    num_classes = 20
    IMAGE_SIZE = 416
    model = YOLOv3(num_classes=num_classes)
    x = torch.randn((2, 3, IMAGE_SIZE, IMAGE_SIZE))
    out = model(x)
    assert model(x)[0].shape == (2, 3, IMAGE_SIZE//32, IMAGE_SIZE//32, num_classes + 5)
    assert model(x)[1].shape == (2, 3, IMAGE_SIZE//16, IMAGE_SIZE//16, num_classes + 5)
    assert model(x)[2].shape == (2, 3, IMAGE_SIZE//8, IMAGE_SIZE//8, num_classes + 5)
    print("Success!")

Overwriting yolov3/model.py
